In [25]:
import os
import sys
# 直接指定完整的專案路徑
project_path = r'C:\Users\archi\Python\Project\tw_stock_analysis'
sys.path.append(project_path)

# 其他必要的導入
import glob
import shutil
import logging
import traceback
import numpy as np
import pandas as pd
import talib
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from utils.config import TWStockConfig, MarketDateRange
from utils.data_utils import MarketDataProcessor

# 設置日誌
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('stock_data_processing.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

In [26]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('stock_data_processing.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

In [27]:
def process_price_data(df):
    """
    處理價格數據，包含資料清理和格式轉換
    """
    try:
        # 複製DataFrame避免修改原始資料
        df = df.copy()
        
        # 處理價格欄位的千分位逗號
        price_columns = ['開盤價', '最高價', '最低價', '收盤價', '最後揭示買價', '最後揭示賣價']
        for col in price_columns:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: str(x).replace(',', '') if isinstance(x, str) else x)
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 處理成交量欄位
        volume_columns = ['成交股數', '成交筆數', '成交金額']
        for col in volume_columns:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: str(x).replace(',', '') if isinstance(x, str) else x)
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 處理日期欄位
        if '日期' in df.columns:
            df['日期'] = pd.to_datetime(df['日期']).dt.strftime('%Y-%m-%d')
            
        # 處理特殊字符
        df = df.replace('--', np.nan)
        df = df.replace('', np.nan)
        
        return df
        
    except Exception as e:
        logging.error(f"Error in process_price_data: {e}")
        logging.error(traceback.format_exc())
        return None

In [28]:
def validate_config():
    """
    驗證必要的目錄和檔案是否存在
    """
    base_path = "D:\\Min\\Python\\Project\\FA_Data"
    required_dirs = [
        "meta_data",
        "technical_analysis",
        "meta_data\\backup"
    ]
    
    try:
        for dir_path in required_dirs:
            full_path = os.path.join(base_path, dir_path)
            if not os.path.exists(full_path):
                os.makedirs(full_path)
                logging.info(f"Created directory: {full_path}")
                
        return True
    except Exception as e:
        logging.error(f"Error validating config: {e}")
        return False

In [29]:
def calculate_and_store_indicators(df, stock_id=None, date=None, by_date=False):
    """Calculate technical indicators and save to files"""
    try:
        # 數據預處理
        df = process_price_data(df)
        if df is None:
            return False
            
        # 檢查數據有效性
        price_columns = ['開盤價', '最高價', '最低價', '收盤價']
        if df[price_columns].isnull().all().any():
            logging.warning(f"Missing price data for {'date '+date if by_date else 'stock ID '+stock_id}")
            return False
            
        # 排序並重設索引
        df = df.sort_values('日期').reset_index(drop=True)
        
        # 使用收盤價計算技術指標
        close_prices = df['收盤價'].values
        high_prices = df['最高價'].values
        low_prices = df['最低價'].values
        
        # 初始化所有技術指標欄位為np.nan
        for col in ['SMA30', 'DEMA30', 'EMA30', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
                   'slowk', 'slowd', 'TSF', 'middleband', 'SAR']:
            df[col] = np.nan
        
        # 計算移動平均線系列
        df['SMA30'] = talib.SMA(close_prices, timeperiod=30)
        df['DEMA30'] = talib.DEMA(close_prices, timeperiod=30)
        df['EMA30'] = talib.EMA(close_prices, timeperiod=30)
        
        # 計算RSI - 修正版本
        try:
            # 直接使用原始的close_prices計算RSI
            rsi = talib.RSI(close_prices, timeperiod=14)
            # 將無限值和NaN替換為50（中性值）
            rsi = np.nan_to_num(rsi, nan=50.0, posinf=100.0, neginf=0.0)
            # 確保值在0-100範圍內
            df['RSI'] = np.clip(rsi, 0, 100)
        except Exception as e:
            logging.warning(f"Error calculating RSI for {stock_id}: {str(e)}")
            df['RSI'] = 50.0  # 錯誤時使用中性值
        
        # 計算MACD
        macd, signal, hist = talib.MACD(close_prices, 
                                      fastperiod=12, 
                                      slowperiod=26, 
                                      signalperiod=9)
        df['MACD'] = macd
        df['MACD_signal'] = signal
        df['MACD_hist'] = hist
        
        # 計算KD指標
        try:
            slowk, slowd = talib.STOCH(high_prices, 
                                     low_prices, 
                                     close_prices,
                                     fastk_period=5, 
                                     slowk_period=3, 
                                     slowk_matype=0,
                                     slowd_period=3, 
                                     slowd_matype=0)
            # 處理可能的無限值和NaN
            slowk = np.nan_to_num(slowk, nan=50.0, posinf=100.0, neginf=0.0)
            slowd = np.nan_to_num(slowd, nan=50.0, posinf=100.0, neginf=0.0)
            df['slowk'] = np.clip(slowk, 0, 100)
            df['slowd'] = np.clip(slowd, 0, 100)
        except Exception as e:
            logging.warning(f"Error calculating KD for {stock_id}: {str(e)}")
            df['slowk'] = 50.0
            df['slowd'] = 50.0
        
        # 計算TSF
        df['TSF'] = talib.TSF(close_prices, timeperiod=14)
        
        # 計算布林通道
        upper, middle, lower = talib.BBANDS(close_prices, 
                                          timeperiod=30, 
                                          nbdevup=2, 
                                          nbdevdn=2, 
                                          matype=0)
        df['middleband'] = middle
        
        # 計算SAR
        try:
            sar = talib.SAR(high_prices, 
                           low_prices, 
                           acceleration=0.02, 
                           maximum=0.2)
            # 設定合理的價格限制
            max_price = df['最高價'].max() * 1.5
            min_price = df['最低價'].min() * 0.5
            sar = np.nan_to_num(sar, nan=df['收盤價'].mean())
            df['SAR'] = np.clip(sar, min_price, max_price)
        except Exception as e:
            logging.warning(f"Error calculating SAR for {stock_id}: {str(e)}")
            df['SAR'] = df['收盤價']  # 使用收盤價作為預設值
        
        # 最後進行一次全面的數值檢查
        technical_columns = ['RSI', 'slowk', 'slowd']
        for col in technical_columns:
            if col in df.columns:
                df[col] = np.clip(df[col], 0, 100)
        
        # 儲存結果
        base_path = "D:\\Min\\Python\\Project\\FA_Data"
        save_path = f"{base_path}\\technical_analysis\\{stock_id}_indicators.csv"
        
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        
        return df
        
    except Exception as e:
        logging.error(f"Error processing stock {stock_id}: {e}")
        logging.error(traceback.format_exc())
        return None

In [30]:
def process_stock_data_batch():
    """批次處理股票資料"""
    try:
        base_path = "D:\\Min\\Python\\Project\\FA_Data"
        stock_data_whole = pd.read_csv(f"{base_path}\\meta_data\\stock_data_whole.csv", low_memory=False)
        
        all_processed_data = []
        grouped = stock_data_whole.groupby('證券代號')
        total_stocks = len(grouped)
        
        processed_count = 0
        failed_stocks = []
        
        # 使用tqdm包裝grouped.items()
        progress_bar = tqdm(grouped, total=total_stocks, 
                          desc="Processing stocks", 
                          unit="stock",
                          bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')
        
        for stock_id, group_df in progress_bar:
            result_df = calculate_and_store_indicators(group_df.copy(), stock_id=stock_id)
            if isinstance(result_df, pd.DataFrame):
                all_processed_data.append(result_df)
                processed_count += 1
                progress_bar.set_postfix({'Success': f"{processed_count}/{total_stocks}"})
            else:
                failed_stocks.append(stock_id)
                progress_bar.set_postfix({'Failed': stock_id})

        progress_bar.close()

        if all_processed_data:
            # 合併所有處理後的資料
            logging.info("Merging processed data...")
            all_stocks_df = pd.concat(all_processed_data, ignore_index=True)
            
            # 備份和儲存
            today = datetime.now().strftime('%Y%m%d')
            backup_dir = f"{base_path}\\meta_data\\backup"
            backup_path = f"{backup_dir}\\all_stocks_data_{today}.csv"
            
            # 保存新的合併檔案
            logging.info("Saving merged data...")
            all_stocks_df.to_csv(f"{base_path}\\meta_data\\all_stocks_data.csv", 
                               index=False, encoding='utf-8-sig')
            
            # 建立備份
            shutil.copy2(f"{base_path}\\meta_data\\all_stocks_data.csv", backup_path)
            
            # 清理舊備份（保留最近7天）
            backup_files = glob.glob(f"{backup_dir}\\all_stocks_data_*.csv")
            backup_files.sort(reverse=True)
            for old_backup in backup_files[7:]:
                os.remove(old_backup)
            
            # 輸出處理結果統計
            logging.info(f"Processing completed:")
            logging.info(f"- Successfully processed: {processed_count}/{total_stocks} stocks")
            if failed_stocks:
                logging.info(f"- Failed stocks: {', '.join(failed_stocks)}")
            
            return True
        else:
            logging.warning("No data was successfully processed")
            return False
            
    except Exception as e:
        logging.error(f"Error in batch processing: {e}")
        logging.error(traceback.format_exc())
        return False

In [31]:
def main():
    """主程序"""
    try:
        config = TWStockConfig()
        date_range = MarketDateRange.last_n_days(45)
        
        logging.info("Starting batch processing...")
        if process_stock_data_batch():
            logging.info("Batch processing completed successfully")
            return True
        else:
            logging.error("Batch processing failed")
            return False
            
    except Exception as e:
        logging.error(f"Error in main process: {e}")
        logging.error(traceback.format_exc())
        return False

In [32]:
if __name__ == "__main__":
    if validate_config():
        success = main()
        if success:
            logging.info("Process completed successfully")
            
            # 顯示一個股票的處理結果作為範例
            example_stock = "2330"
            try:
                df = pd.read_csv(f"D:\\Min\\Python\\Project\\FA_Data\\technical_analysis\\{example_stock}_indicators.csv")
                print(f"\n{example_stock} 技術指標計算結果範例:")
                print(df.tail())
            except Exception as e:
                logging.error(f"Error showing example results: {e}")
    else:
        logging.error("Configuration validation failed")

2024-11-12 15:06:31,519 - INFO - Starting batch processing...
Processing stocks: 100%|████████████████████████████████████████████████████████| 1148/1148 [01:22<00:00, 13.91stock/s]
2024-11-12 15:07:59,382 - INFO - Merging processed data...
2024-11-12 15:08:00,059 - INFO - Saving merged data...
2024-11-12 15:08:57,363 - INFO - Processing completed:
2024-11-12 15:08:57,364 - INFO - - Successfully processed: 1147/1148 stocks
2024-11-12 15:08:57,365 - INFO - - Failed stocks: 911201
2024-11-12 15:08:57,633 - INFO - Batch processing completed successfully
2024-11-12 15:08:57,635 - INFO - Process completed successfully



2330 技術指標計算結果範例:
      證券代號 證券名稱      成交股數   成交筆數         成交金額     開盤價     最高價     最低價     收盤價  \
2588  2330  台積電  59404980  58611  63110184620  1050.0  1080.0  1040.0  1060.0   
2589  2330  台積電  46030563  41583  48979606039  1050.0  1075.0  1050.0  1065.0   
2590  2330  台積電  38840413  58203  42224017162  1085.0  1090.0  1080.0  1090.0   
2591  2330  台積電  36630666  48859  39467913757  1080.0  1090.0  1070.0  1085.0   
2592  2330  台積電  54098799  92756  57135037210  1060.0  1065.0  1050.0  1050.0   

     漲跌(+/-)  ...        EMA30        RSI       MACD  MACD_signal  MACD_hist  \
2588       +  ...  1026.622218  58.984721  14.997919    19.421178  -4.423259   
2589       +  ...  1029.098204  60.184468  15.371479    18.611238  -3.239759   
2590       +  ...  1033.027352  65.602329  17.483283    18.385647  -0.902364   
2591       -  ...  1036.380426  63.734385  18.539730    18.416464   0.123266   
2592       -  ...  1037.259109  52.471476  16.364129    18.005997  -1.641868   

          slow